

## Project Overview

This notebook focuses on understanding **loss functions** and their role in
training neural networks. Loss functions quantify how far a model’s predictions
are from the true values and provide the signal used to update model parameters
during training.

Rather than treating loss functions as black-box formulas, this project aims to
build intuition by implementing and comparing commonly used loss functions in
neural networks.

---

## Why Loss Functions Matter

A neural network learns by minimizing a loss function. The choice of loss
function directly affects:

- How errors are penalized
- How gradients are computed
- How quickly and stably the model converges

Using an inappropriate loss function can slow down learning or lead to poor
model performance, even if the network architecture is well designed.

---




## Mean Absolute Error (MAE)

Mean Absolute Error (MAE) measures the average absolute difference between the
true values and the predicted values.

It treats all errors equally, regardless of their direction or magnitude.

---

## Intuition

- Measures how far predictions are from actual values on average
- Every error contributes linearly to the loss
- Less sensitive to large outliers compared to MSE

---

## When to Use MAE

- Regression problems
- When robustness to outliers is important
- When absolute differences are easier to interpret

---

## Limitations

- Gradient is constant, which can make optimization slower
- Does not strongly penalize large errors

---


In [ ]:
import numpy as np

y_predicted = np.array([1,1,0,0,1])
y_true = np.array([0.3,0.7,1,0,0.5])

#### Implementation of mean absolute error

In [ ]:
def mae(y_predicted, y_true):
  total_error = 0
  for yp, yt in zip(y_predicted, y_true):
    total_error += abs(yp - yt)
  print('Total error is:', total_error)
  mae = total_error/len(y_predicted)
  print('Mean absolute error is:', mae)
  return mae

In [ ]:
mae(y_predicted,y_true)

Total error is: 2.5
Mean absolute error is: 0.5


np.float64(0.5)

In [ ]:
## By using numpy

def mae_np(y_predicted, y_true):
  return np.mean(np.abs(y_predicted - y_true))

In [ ]:
mae_np(y_predicted,y_true)

np.float64(0.5)

## Mean Squared Error (MSE)

Mean Squared Error (MSE) calculates the average of the squared differences
between predicted values and actual values.

By squaring the errors, MSE penalizes larger errors more heavily than smaller
ones.

---

## Intuition

- Small errors contribute less to the loss
- Large errors dominate the loss value
- Encourages the model to focus on correcting big mistakes

---

## When to Use MSE

- Regression problems
- When large errors are particularly undesirable
- Commonly used in training neural networks

---

## Limitations

- Sensitive to outliers due to squared term
- Loss values are not directly interpretable in original units

---


In [ ]:
def mse(y_predicted, y_true):
  total_error = 0
  for yp, yt in zip(y_predicted, y_true):
    total_error += (yt-yp)**2
  print('Total error is:',total_error)
  mse = total_error/len(y_true)
  print('Mean squared error is:',mse)
  return mse

In [ ]:
mse(y_predicted,y_true)

Total error is: 1.83
Mean squared error is: 0.366


np.float64(0.366)

In [ ]:
## By using numpy

np.mean(np.square(y_true-y_predicted))

np.float64(0.366)

## Binary Cross-Entropy Loss

Binary Cross-Entropy measures the difference between true binary labels and
predicted probabilities.

It is widely used in binary classification problems where outputs represent
probabilities between 0 and 1.

---

## Intuition

- Confident correct predictions are rewarded
- Confident wrong predictions are heavily penalized
- Encourages probabilistic correctness, not just accuracy

---

## When to Use Binary Cross-Entropy

- Binary classification problems
- When the model output is a probability (sigmoid activation)
- Logistic regression and neural network classifiers

---

## Limitations

- Sensitive to incorrect confident predictions
- Requires outputs to be properly bounded between 0 and 1



In [ ]:
# True labels (ground truth)
y_true = [1, 0, 1, 1, 0]

# Predicted probabilities from model (sigmoid output)
y_pred = [0.9, 0.2, 0.8, 0.6, 0.1]

In [ ]:
def bce(y_true, y_pred, epsilon=1e-15):
    """
    Computes Binary Cross-Entropy loss from scratch.

    Parameters:
    y_true : array-like (true labels: 0 or 1)
    y_pred : array-like (predicted probabilities between 0 and 1)
    epsilon: small value to avoid log(0)

    Returns:
    loss : scalar (mean BCE loss)
    """

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    return loss


In [ ]:
loss = bce(y_true, y_pred)
print(f"Binary Cross-Entropy Loss: {loss:.4f}")

Binary Cross-Entropy Loss: 0.2336


## Categorical Cross-Entropy Loss

Categorical Cross-Entropy (CCE) is a loss function used for **multi-class
classification problems**, where the target variable can belong to one of more
than two classes.

It compares the true class distribution with the predicted probability
distribution produced by the model.

---

## Why Categorical Cross-Entropy Is Needed

Binary Cross-Entropy is limited to two classes. When dealing with multiple
classes (e.g., digits 0–9 in MNIST), a loss function is required that can handle
probability distributions across all classes.

Categorical Cross-Entropy fulfills this role by penalizing incorrect probability
assignments across all classes.

---

## How It Works (Intuition)

- The true label is represented as a **one-hot encoded vector**
- The model outputs a probability distribution (usually via **Softmax**)
- The loss measures how much probability the model assigns to the correct class

If the model assigns high probability to the correct class, the loss is low.
If it assigns low probability, the loss increases sharply.

---

## When to Use Categorical Cross-Entropy

- Multi-class classification problems
- When class labels are one-hot encoded
- When the output layer uses Softmax activation

---

## Limitations

- Requires one-hot encoded labels
- Sensitive to confident wrong predictions
- Numerically unstable without proper clipping


In [ ]:
def softmax(logits):
  """
    Computes softmax probabilities in a numerically stable way.
    """
  exp_values = np.exp(logits - np.max(logits, axis=1, keepdims=True))
  return exp_values / np.sum(exp_values, axis=1, keepdims=True)

In [ ]:
def categorical_cross_entropy(y_true, y_pred, epsilon=1e-15):
    """
    Computes Categorical Cross-Entropy loss from scratch.

    Parameters:
    y_true : array (one-hot encoded true labels)
    y_pred : array (predicted probabilities from softmax)
    epsilon: small value to avoid log(0)

    Returns:
    loss : scalar (mean CCE loss)
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Clip predictions to avoid log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    # Compute loss
    loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))
    return loss


In [ ]:
# True labels (one-hot encoded)
y_true = np.array([
    [1, 0, 0],  # Class 0
    [0, 1, 0],  # Class 1
    [0, 0, 1]   # Class 2
])

# Raw model outputs (logits)
logits = np.array([
    [3.0, 1.0, 0.2],
    [0.5, 2.5, 0.3],
    [0.1, 0.2, 2.8]
])

# Convert logits to probabilities
y_pred = softmax(logits)


In [ ]:
loss = categorical_cross_entropy(y_true, y_pred)
print(f"Categorical Cross-Entropy Loss: {loss:.4f}")


Categorical Cross-Entropy Loss: 0.1772


## Conclusion

In this notebook, different loss functions were explored to understand how neural
networks quantify errors and learn from data.

The following loss functions were implemented and analyzed:

- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Binary Cross-Entropy
- Categorical Cross-Entropy

Through these implementations, it became clear that loss functions are not just
mathematical formulas, but learning signals that directly influence optimization
and model behavior.

---

## Key Learnings

- Regression problems require loss functions like MAE or MSE
- Classification problems require probabilistic loss functions such as
  binary or categorical cross-entropy
- Confident wrong predictions are penalized heavily in cross-entropy losses
- The choice of loss function must align with the task and model output

